# Snowpark ML Model Registry

## Notebook 03 - Publishing and Deploying a Model


Now that we have a model pickled to a local filesystem, let's explore how to publish this model to the Snowpark Model Registry.

### Steps:
1. Setup (preamble)
2. Create a Model Registry
3. Log the Model in the Model Registry
4. Use the Registered Model to Make Preditions

### 1. Setup (preamble)

In [ ]:
# Snowpark for Python
from snowflake.snowpark import Session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark.types import StructType, StructField, DoubleType, StringType, DecimalType
from snowflake.snowpark.functions import *
from snowflake.snowpark.types import *

# Snowpark ML
import snowflake.ml.modeling.preprocessing as snowparkml
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.metrics.correlation import correlation
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import *

# General Data Science Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
import json
import joblib

# Warning Suppression
import warnings; warnings.simplefilter('ignore')

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

# current_user = session.get_current_user()
current_user = props['user']

- Load relevant work from the previous notebooks

In [ ]:
# Load the grid search object
grid_search = joblib.load('grid_search.joblib')

# Extract the best model from the grid search
tuned_model = grid_search.to_sklearn().best_estimator_

# Get the training dataset table persisted from the first notebook
trainDF = session.table(current_user + '_db.public.training_data')

# Get test dataset table persisted from the first notebook
testDF = session.table(current_user + '_db.public.test_data')

# Define some lists of convenience
FEATURE_COLUMNS = trainDF.drop("CHURNED").columns
LABEL_COLUMNS = ["CHURNED"]
OUTPUT_COLUMNS = ["PREDICTED_CHURN"]

### 2. Create a Model Registry

#### Model Registry

Snowpark ML Operations allows you to manage models regardless of origin!

The model registry allows you to register, manage and use several types of machine learning models created both within and outside of Snowflake.

The registry is designed to be a first-class schema-level object in Snowflake that provides a version container of ML model artifacts with full RBAC (role-based access control) support and APIs for Python and SQL.

<img src="images/Snowpark_ML_Operations.png" alt="MLAPIQuery" style="width:65%;display:block;margin-left:10%;" />

Snowpark Model Registry provides a model versioning and deployment framework. This allows us to log models, tag parameters and metrics, track metadata, create versions, and ultimately deploy models into a Snowflake warehouse or Snowpark Container Service for batch scoring tasks.  All of this is achieved inside a Snowflake Notebook without data leaving the governance boundaries of Snowflake.

The Snowpark Model Registry includes support for the following types of models:

- [Snowpark ML Modeling](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-modeling)
- scikit-learn
- XGBoost
- PyTorch
- TensorFlow
- MLFlow PyFunc
- HuggingFace pipeline

#### Create a schema for the registry 

In [ ]:
REGISTRY_SCHEMA = 'ml_registry'
_ = (session.sql('create schema if not exists ' + current_user + '_db.' + REGISTRY_SCHEMA).
     collect()
    )

The model registry will be associated with this database and schema. Snowflake recommends creating a dedicated schema for this purpose, and if you wish to have multiple registries, you must provide a separate schema for each. 

#### Define (create) the registry

In [ ]:
from snowflake.ml.registry import Registry

# Create a registry and log the model
registry = Registry(session=session, database_name=current_user + '_db', schema_name='ml_registry')

### 3. Log the Model in the Model Registry

Adding a model to the registry is called *logging*.

The function `snowflake.ml.Registry.log_model(...)` takes the following aguments:
- **model** - Model object of supported types such as Scikit-learn, XGBoost, Snowpark ML, PyTorch, TorchScript, Tensorflow, Tensorflow Keras, MLFlow, HuggingFace Pipeline, or Custom Model.
- **model_name** - Name to identify the model.
- **version_name** - Version identifier for the model. Combination of model_name and version_name must be unique.
- **comment** - Comment associated with the model version. Defaults to None.
- **metrics** - A JSON serializable dictionary containing metrics linked to the model version. Defaults to None.
- **signatures** - Model data signatures for inputs and outputs for various target methods. Defaults to None.
- **sample_input_data** - Sample input data to infer model signatures from. Defaults to None.
- **conda_dependencies** - List of Conda package specifications. Defaults to None.
- **pip_requirements** - List of Pip package specifications. Defaults to None.
- **python_version** - Python version in which the model is run. Defaults to None.
- **code_paths** - List of directories containing code to import. Defaults to None.
- **ext_modules** List of external modules to pickle with the model object. Only supported when logging the following types of model: Scikit-learn, Snowpark ML, PyTorch, TorchScript and Custom Model. Defaults to None.
- **options** - Optional. Additional model saving options.

Logging a model by calling the registry’s `log_model` method will:
- Serialize the model, a Python object, and create a Snowflake model object from it.
- Add metadata such as a comment to the model as specified in the `log_model` call.
- (Object tagging is done independently of the log_model() method)

In [ ]:
# Define model name and version
model_name = 'CHURN_PREDICTION'
version_name = 'V1'

# Get sample input data to pass into the registry logging function below
train_features = trainDF.select(FEATURE_COLUMNS).limit(100)

# Clean up existing model if we're re-running this cell
_ = session.sql('USE SCHEMA ' + REGISTRY_SCHEMA).collect()
_ = session.sql('DROP MODEL IF EXISTS ' + model_name).collect()

mv = registry.log_model(
    tuned_model, # The Python model object we want to log into the registry
    model_name=model_name, # model_name needs to conform to Snowflake identifier syntax
    version_name=version_name, # version_name also needs to conform to Snowflake indentifier syntax
    sample_input_data=train_features,
    conda_dependencies = ["xgboost==1.7.3"]
)

# mv is a ModelVersion object; see https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/api/model/snowflake.ml.model.ModelVersion

#### Show models

List the registered models in the registry.

In [ ]:
# This will return all the models in the registry
registry.show_models()

In [ ]:
# This will show details for the version(s) of a specific model
registry.get_model(model_name).show_versions()

In [ ]:
# Let's add a comment and some additional metrics to this version
# Be aware these metrics may not match the metrics obtained during testing in the previous notebook
#
# You might use this metadata to document the performance drift of a deployed model

mv.comment = 'This is the first version of our classifier!'
mv.set_metric(metric_name = 'accuracy', value = 0.87)
mv.set_metric(metric_name = 'f1', value = 0.619)

#### Retrieve the model into a new object
This verifies persistence of the model. 

**NOTE:** In Snowpark Model Registry, you will not see any objects in the database schema for your registry. Persistence is handled "under the hood" by the cloud services layer of the system. However, the schema must exist for your registry.

In [ ]:
# The model 
retrieved_model = registry.get_model(model_name)

# The specific version of the model
retrieved_model_version = retrieved_model.version('v1')

In [ ]:
# Access the comment
print(retrieved_model_version.comment)
print(retrieved_model_version.description) # Both description and comment are accessors for the comment metadata

In [ ]:
# All metrics
retrieved_model_version.show_metrics()

In [ ]:
# A single metric
retrieved_model_version.get_metric('accuracy')

#### 4. Use the Registered Model to Make Predictions

Now we will do a batch scoring of the test data.  Note that inferencing happens on the remote Snowflake warehouse. 

(Feel free to switch over to Snowsight and look at the Query History to get a sense of what happens in the back end.)

In [ ]:
result = retrieved_model_version.run(testDF, function_name = 'predict')
(result.
 withColumnRenamed('"output_feature_0"', 'prediction').
 select('prediction', 'churned').
 show()
)

In [ ]:
session.close()